# Softmax Regression: multiclass digit classification

This notebook uses scikit-learn's `LogisticRegression` with `multi_class='multinomial'`, which is Softmax Regression. It predicts one of the ten digit classes (0–9).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, log_loss
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

digits = load_digits()
X, y = digits.data, digits.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scaling is recommended because Logistic Regression uses an iterative solver and L2 regularization.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(multi_class='multinomial', max_iter=2000, random_state=42)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
y_proba = model.predict_proba(X_test_scaled)

print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print(f'Log loss: {log_loss(y_test, y_proba):.4f}')
print('\nClassification report:\n', classification_report(y_test, y_pred))

ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred), display_labels=digits.target_names).plot(cmap='Blues')
plt.title('Softmax Regression: confusion matrix')
plt.show()

## Model, loss, scaling, and evaluation

### Formula notation

- $X \in \mathbb{R}^{n \times d}$: real-valued feature matrix with $n$ examples and $d=64$ pixel features.
- $\mathbf{x}_i$: features for digit image $i$; $y_i \in \{0, \ldots, 9\}$: its true class.
- $K=10$: number of classes; $\mathbf{W} \in \mathbb{R}^{K \times d}$: learned weight matrix; $\mathbf{b} \in \mathbb{R}^{K}$: learned bias vector.
- $z_{ik}$: linear score (logit) for example $i$ and class $k$; $p_{ik}$: predicted probability of that class.

### Softmax model

$$z_{ik} = \mathbf{w}_k^T\mathbf{x}_i + b_k$$

$$p_{ik} = \frac{e^{z_{ik}}}{\sum_{j=1}^{K} e^{z_{ij}}}$$

$$\hat{y}_i = \operatorname*{argmax}_{k} p_{ik}$$

Softmax is used because the ten classes are mutually exclusive. It converts ten linear scores into non-negative probabilities that sum to $1$, then chooses the most probable digit.

### Training objective: multiclass cross-entropy

$$J = -\frac{1}{n}\sum_{i=1}^{n} \log(p_{i,y_i})$$

This loss penalizes the model when it gives low probability to the true digit. **Minimize** it: $0$ is best and there is no fixed worst value.

### Feature scaling

$$x_{ij}^{\mathrm{scaled}} = \frac{x_{ij}-\mu_j}{\sigma_j}$$

Calculate $\mu_j$ and $\sigma_j$ from training data only, then transform test data with those same values. Scaling helps the iterative solver and L2 regularization; fitting the scaler only on training data prevents leakage.

### Evaluation

$$\mathrm{Accuracy} = \frac{1}{m}\sum_{i=1}^{m}\mathbb{1}(\hat{y}_i=y_i)$$

**Maximize accuracy:** its range is $[0,1]$, where $1$ is best. The confusion matrix shows actual classes by row and predicted classes by column. `classification_report` gives precision, recall, and $F_1$ for each digit; maximize all three, where $1$ is best.

## Softmax Regression from scratch (NumPy only)

The implementation below mirrors the from-scratch Logistic Regression in the companion notebook.
It uses only NumPy for: stratified train/test split, manual feature scaling, softmax regression
trained with gradient descent, and hand-computed evaluation metrics.

In [ ]:
import numpy as np
from sklearn.datasets import load_digits   # only for loading the data

# ── Load data ────────────────────────────────────────────────────────
digits = load_digits()
X, y = digits.data, digits.target   # X: (1797, 64), y: 0–9

# ── Stratified train/test split (pure NumPy) ─────────────────────────
def stratified_split_numpy(X, y, test_size=0.2, seed=42):
    np.random.seed(seed)
    train_idx, test_idx = [], []

    for label_value in np.unique(y):
        class_indices = np.where(y == label_value)[0]
        shuffled = np.random.permutation(class_indices)
        n_test = int(len(shuffled) * test_size)
        test_idx.extend(shuffled[:n_test])
        train_idx.extend(shuffled[n_test:])

    train_idx = np.random.permutation(train_idx)
    test_idx  = np.random.permutation(test_idx)

    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

X_train, X_test, y_train, y_test = stratified_split_numpy(X, y, test_size=0.2, seed=42)

# ── Manual feature scaling (fit on train only) ───────────────────────
def fit_scaler(X):
    mean = X.mean(axis=0)
    std  = X.std(axis=0)
    std[std == 0] = 1.0          # avoid division by zero for constant features
    return mean, std

def apply_scaler(X, mean, std):
    return (X - mean) / std

mean, std = fit_scaler(X_train)
X_train_scaled = apply_scaler(X_train, mean, std)
X_test_scaled  = apply_scaler(X_test, mean, std)

# ── Helper: add bias column ──────────────────────────────────────────
def add_bias(X):
    return np.hstack([np.ones((X.shape[0], 1)), X])

# ── Softmax function (numerically stable) ────────────────────────────
def softmax(Z):
    """Row-wise softmax.  Z: (n, K) -> probabilities (n, K)."""
    exp_Z = np.exp(Z - Z.max(axis=1, keepdims=True))   # subtract row-max for stability
    return exp_Z / exp_Z.sum(axis=1, keepdims=True)

# ── One-hot encoding ─────────────────────────────────────────────────
def one_hot(y, K):
    """Convert label vector y (n,) to one-hot matrix (n, K)."""
    oh = np.zeros((len(y), K))
    oh[np.arange(len(y)), y] = 1.0
    return oh

# ── Softmax regression: fit with gradient descent ────────────────────
def fit_softmax_regression(X, y, lr=0.1, n_iters=1000):
    """
    Gradient descent for multiclass cross-entropy.

    Parameters
    ----------
    X : array of shape (n, d) – already scaled features.
    y : array of shape (n,)   – integer class labels 0..K-1.
    lr : learning rate.
    n_iters : number of gradient-descent iterations.

    Returns
    -------
    W : array of shape (d+1, K) – learned weight matrix (first row = biases).
    """
    X_b = add_bias(X)                       # (n, d+1)
    n_samples, n_features = X_b.shape
    K = len(np.unique(y))
    Y_oh = one_hot(y, K)                    # (n, K)
    W = np.zeros((n_features, K))           # (d+1, K)

    for i in range(n_iters):
        Z = X_b @ W                         # (n, K) logits
        P = softmax(Z)                      # (n, K) probabilities

        # gradient of cross-entropy w.r.t. W
        gradient = (1 / n_samples) * X_b.T @ (P - Y_oh)   # (d+1, K)
        W -= lr * gradient

        if i % 200 == 0:
            # cross-entropy loss
            loss = -np.mean(np.log(P[np.arange(n_samples), y] + 1e-9))
            print(f'Iter {i:4d}: loss = {loss:.4f}')

    return W

# ── Predict helpers ──────────────────────────────────────────────────
def predict_proba_softmax(X, W):
    return softmax(add_bias(X) @ W)

def predict_softmax(X, W):
    return np.argmax(predict_proba_softmax(X, W), axis=1)

# ── Train ────────────────────────────────────────────────────────────
W = fit_softmax_regression(X_train_scaled, y_train, lr=0.1, n_iters=1000)

# ── Predict ──────────────────────────────────────────────────────────
y_pred = predict_softmax(X_test_scaled, W)
y_proba = predict_proba_softmax(X_test_scaled, W)

# ── Manual metrics ───────────────────────────────────────────────────
accuracy = np.mean(y_pred == y_test)
cross_ent = -np.mean(np.log(y_proba[np.arange(len(y_test)), y_test] + 1e-9))

# Per-class precision, recall, F1
K = len(np.unique(y_test))
print(f'\nAccuracy:        {accuracy:.4f}')
print(f'Cross-entropy:   {cross_ent:.4f}')
print(f'\n{"Class":>5}  {"Precision":>9}  {"Recall":>6}  {"F1":>6}  {"Support":>7}')
print('-' * 44)
for k in range(K):
    tp = np.sum((y_pred == k) & (y_test == k))
    fp = np.sum((y_pred == k) & (y_test != k))
    fn = np.sum((y_pred != k) & (y_test == k))
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    support = int(np.sum(y_test == k))
    print(f'{k:5d}  {prec:9.4f}  {rec:6.4f}  {f1:6.4f}  {support:7d}')

# Confusion matrix
cm = np.zeros((K, K), dtype=int)
for true, pred in zip(y_test, y_pred):
    cm[true, pred] += 1
print(f'\nConfusion matrix (rows = actual, cols = predicted):\n{cm}')